# NAS - Optuna

- **Authored by:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

## 1. Setup and Configuration

### 1.1. Environment Variables

In [ ]:
import os

# Async CUDA allocator
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'

# If cuDNN autotune fails, fall back to a safe (but slower) algorithm.
os.environ["XLA_FLAGS"] = "--xla_gpu_strict_conv_algorithm_picker=false"

# Allow TensorFlow to allocate GPU memory as needed
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true' 

### 1.2. Imports

In [ ]:
from _imports import * # Centralized file containing all imports

### 1.3. GPU Management

In [ ]:
# Specify GPU to use (e.g., GPU 0)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
troo.get_gpu_info()

## 2. Run Parameters 

In [ ]:
NUM_TRIALS = 500
EPOCHS = 50
TOP_K = 10  # Number of top trials to save

mixed_precision.set_global_policy("mixed_float16")

#? Set to an existing path to resume training
RESUME_TRAINING_PATH = "runs/nas_lidar_flat_sig" # None or "runs/nas_1" 

In [ ]:
RUN_DIR = RESUME_TRAINING_PATH or troo.create_run_directory(prefix="nas_")
print(f"Run directory: {RUN_DIR}")

## 3. Data Loading and Preprocessing

In [ ]:
def convert_to_multihot_labels(y: np.ndarray, k: int = 5) -> np.ndarray:
    """
    Converts beam score targets (shape: [N, 8, 32]) into multi-hot vectors (shape: [N, 256]),
    setting the k highest scores to 1 and the rest to 0.
    """
    # Flatten each sample to length 256
    y_flat = y.reshape(y.shape[0], -1)  # (N, 256)
    
    N, M = y_flat.shape
    multi_hot = np.zeros((N, M), dtype=np.float32)
    
    # Find indices of the k largest values for each sample
    topk_idx = np.argpartition(y_flat, -k, axis=1)[:, -k:]
    for i in range(N):
        multi_hot[i, topk_idx[i]] = 1.0
        
    return multi_hot

In [ ]:
# Define the base directory for data files
DATA_DIR = "./data/s008"
TOP_K = 5  # Number of top scores to consider for multi-hot encoding

# —————————————————————————————— Load Train Data ————————————————————————————— #
beam_output_train_path = os.path.join(DATA_DIR, "beam_output", "beam_output_train.npz")
coord_input_train_path = os.path.join(DATA_DIR, "coord_input", "coord_train.npz")
lidar_input_train_path = os.path.join(DATA_DIR, "lidar_input", "lidar_train.npz")

# Load the data from the .npy files
s008_y_train = np.load(beam_output_train_path)['output_classification']
s008_coord_input = np.load(coord_input_train_path)['coordinates']
s008_lidar_input = np.load(lidar_input_train_path)['input']

# Cast target beam outputs to float - REMOVING USELESS IMAG PART
s008_y_train = s008_y_train.astype(np.float32)
s008_coord_input = s008_coord_input.astype(np.float32)

print(f"Shape before conversion: {s008_y_train.shape}")
s008_y_train = convert_to_multihot_labels(s008_y_train, k=TOP_K)
print(f"Shape after conversion: {s008_y_train.shape}")

# Print the shapes of the loaded data
print(f"y_train shape: {s008_y_train.shape}")
print(f"coord_input shape: {s008_coord_input.shape}")
print(f"lidar_input shape: {s008_lidar_input.shape}")

# ——————————————————————————— Load Validation Data ——————————————————————————— #
beam_output_val_path = os.path.join(DATA_DIR, "beam_output", "beam_output_val.npz")
coord_input_val_path = os.path.join(DATA_DIR, "coord_input", "coord_val.npz")
lidar_input_val_path = os.path.join(DATA_DIR, "lidar_input", "lidar_val.npz")

# Load the data from the .npy files
s008_y_val = np.load(beam_output_val_path)['output_classification']
s008_coord_input_val = np.load(coord_input_val_path)['coordinates']
s008_lidar_input_val = np.load(lidar_input_val_path)['input']

# Cast target beam outputs to float - REMOVING USELESS IMAG PART
s008_y_val = s008_y_val.astype(np.float32)
s008_coord_input_val = s008_coord_input_val.astype(np.float32)

print(f"Shape before conversion: {s008_y_val.shape}")
s008_y_val = convert_to_multihot_labels(s008_y_val, k=TOP_K)
print(f"Shape after conversion: {s008_y_val.shape}")

# Print the shapes of the loaded data
print(f"y_val shape: {s008_y_val.shape}")
print(f"coord_input_val shape: {s008_coord_input_val.shape}")
print(f"lidar_input_val shape: {s008_lidar_input_val.shape}")

# —————————————————————— Merge train and validation data ————————————————————— #
s008_y_train = np.concatenate((s008_y_train, s008_y_val), axis=0)
s008_coord_input = np.concatenate((s008_coord_input, s008_coord_input_val), axis=0)
s008_lidar_input = np.concatenate((s008_lidar_input, s008_lidar_input_val), axis=0)

# Print the shapes of the merged data
print(f"y_train shape: {s008_y_train.shape}")
print(f"coord_input shape: {s008_coord_input.shape}")
print(f"lidar_input shape: {s008_lidar_input.shape}")

In [ ]:
# Define the base directory for data files
DATA_DIR = "./data/s009"

# Construct full paths to the .npy files
beam_output_path = os.path.join(DATA_DIR, "beam_output", "beam_output.npz")
coord_input_path = os.path.join(DATA_DIR, "coord_input", "coord_input.npz")
lidar_input_path = os.path.join(DATA_DIR, "lidar_input", "lidar_input.npz")

# Load the data from the .npy files
s009_y = np.load(beam_output_path)['output_classification']
s009_coord_input = np.load(coord_input_path)['coordinates']
s009_lidar_input = np.load(lidar_input_path)['input']

# Cast target beam outputs to float - REMOVING USELESS IMAG PART
s009_y = s009_y.astype(np.float32)
s009_coord_input = s009_coord_input.astype(np.float32)

print(f"Shape before conversion: {s009_y.shape}")
s009_y = convert_to_multihot_labels(s009_y, k=TOP_K)
print(f"Shape after conversion: {s009_y.shape}")

# Print the shapes of the loaded data
print(f"y shape: {s009_y.shape}")
print(f"coord_input shape: {s009_coord_input.shape}")
print(f"lidar_input shape: {s009_lidar_input.shape}")

In [ ]:
# # ----------------------- Subsample dataset for testing ---------------------- #
# SAMPLE_SIZE = 50  # Use a subset of x samples for testing
# s008_y_train = s008_y_train[:SAMPLE_SIZE]
# s008_coord_input = s008_coord_input[:SAMPLE_SIZE]
# s008_image_input = s008_image_input[:SAMPLE_SIZE]
# s008_lidar_input = s008_lidar_input[:SAMPLE_SIZE]

# # Print the shapes of the loaded data
# print(f"y_train s008 shape: {s008_y_train.shape}")
# print(f"coord_input s008 shape: {s008_coord_input.shape}")
# print(f"image_input s008 shape: {s008_image_input.shape}")
# print(f"lidar_input s008 shape: {s008_lidar_input.shape}")

# s009_y_train = s009_y_train[:SAMPLE_SIZE]
# s009_coord_input = s009_coord_input[:SAMPLE_SIZE]
# s009_image_input = s009_image_input[:SAMPLE_SIZE]
# s009_lidar_input = s009_lidar_input[:SAMPLE_SIZE]

# print(f"y_train s009 shape: {s009_y_train.shape}")
# print(f"coord_input s009 shape: {s009_coord_input.shape}")
# print(f"image_input s009 shape: {s009_image_input.shape}")
# print(f"lidar_input s009 shape: {s009_lidar_input.shape}")

## 4. Getters

### 4.1. Regularizers

In [ ]:
def get_regularizer(trial: optuna.Trial, name: str) -> Optional[tf.keras.regularizers.Regularizer]:
    """
    Suggests a regularization strategy using Optuna and returns the corresponding Keras regularizer.
    
    Args:
        trial (optuna.Trial): Optuna trial object used to sample the regularizer.
        name (str): Unique identifier for this regularizer parameter (used as key).

    Returns:
        Optional[tf.keras.regularizers.Regularizer]: The selected Keras regularizer instance,
        or `None` if "none" was selected.
    """
    # Suggest a regularizer type
    reg_type: str = trial.suggest_categorical(
        name,
        [
            "none",
            "l1",
            "l2",
            "l1l2",
            # "orthogonal",  #! only works for rank-2 tensors
        ],
    )

    # Map each regularizer name to a corresponding Keras regularizer instance
    regularizer_map: Dict[str, Optional[tf.keras.regularizers.Regularizer]] = {
        "none": None,
        "l1": regularizers.L1(l1=0.01),
        "l2": regularizers.L2(l2=0.01),
        "l1l2": regularizers.L1L2(l1=0.01, l2=0.01),
        "orthogonal": regularizers.OrthogonalRegularizer(factor=0.01, mode="rows"),
    }

    # Return the appropriate regularizer, or None if not found
    return regularizer_map.get(reg_type, None)

### 4.2. Activation Functions

In [ ]:
def get_activation(trial: Any, name: str) -> Union[str, Callable[..., layers.Layer]]:
    """
    Suggests an activation function from a predefined list using Optuna.

    Args:
        trial (Any): The Optuna trial instance used to suggest a value.
        name (str): A unique name for this hyperparameter (e.g., "layer_1_activation").

    Returns:
        Union[str, Callable[..., layers.Layer]]: A string representing the activation function.
        This can be passed directly into a Keras layer's `activation=` argument.
    """
    return trial.suggest_categorical(
        name,
        [
            "relu",
            "tanh",
            "sigmoid",  # Logistic
            "elu", 
            "swish",  # x * sigmoid(x)
            "leaky_relu",
        ],
    )

### 4.3. Optimizers

In [ ]:
def get_optimizer(trial: optuna.Trial) -> tf.keras.optimizers.Optimizer:
    """
    Suggests and returns a TensorFlow optimizer with a trial-based learning rate.

    Args:
        trial (optuna.Trial): Optuna trial object used for hyperparameter suggestion.

    Returns:
        tf.keras.optimizers.Optimizer: An instance of the selected optimizer.
    """
    # Suggest optimizer name from a predefined categorical set
    optimizer_name = trial.suggest_categorical(
        "optimizer",
        [
            "AdamW",
            "SGD",
            "Adam",
            "RMSprop",
            "Nadam",
            "Lion",
        ],
    )

    # Suggest learning rate on a logarithmic scale between 1e-5 and 1e-2
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)

    # Mapping of optimizer names to their TensorFlow classes
    optimizer_map: Dict[str, Type[tf.keras.optimizers.Optimizer]] = {
        "Adam": optimizers.Adam,
        "AdamW": optimizers.AdamW,
        "SGD": optimizers.SGD,
        "RMSprop": optimizers.RMSprop,
        "Nadam": optimizers.Nadam,
        "Lion": optimizers.Lion,
    }

    # Raise error if selected optimizer is not supported in the current context
    if optimizer_name not in optimizer_map:
        raise ValueError(
            f"Optimizer '{optimizer_name}' is not supported. "
            f"Supported optimizers are: {list(optimizer_map.keys())}."
        )

    # Instantiate and return the selected optimizer with suggested learning rate
    return optimizer_map[optimizer_name](learning_rate=learning_rate)

### 4.4. Callbacks

In [ ]:
def get_callbacks(trial: optuna.Trial, checkpoint_dir: str) -> List[tf.keras.callbacks.Callback]:
    """
    Constructs and returns a list of Keras callbacks tailored for Optuna trials.

    Args:
        trial (optuna.Trial): The current Optuna trial object.
        checkpoint_dir (str): Directory where model weights will be saved.

    Returns:
        List[tf.keras.callbacks.Callback]: A list of callbacks to pass into `model.fit()`.
    """
    # Construct path for saving weights for this specific trial
    checkpoint_path: str = os.path.join(checkpoint_dir, f"trial_{trial.number}.weights.h5")

    # Metric to monitor for early stopping and checkpointing
    monitor: str = "val_loss"

    # Stop training early if no improvement in validation loss for N epochs
    early_stopping = callbacks.EarlyStopping(
        monitor=monitor,
        patience=6,  # number of epochs to wait
        restore_best_weights=True,
        verbose=1,
    )

    # Reduce learning rate if validation loss plateaus
    reduce_lr = callbacks.ReduceLROnPlateau(
        monitor=monitor,
        patience=3,  # how many epochs to wait before reducing LR
        factor=0.2,  # reduce LR by this factor
        min_lr=1e-6,  # don't reduce below this
        verbose=1,
    )

    # Save only the best model weights based on monitored metric
    model_checkpoint = callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor=monitor,
        save_best_only=True,  # only save weights if val_loss improves
        save_weights_only=True,  # save only the weights (not full model)
        verbose=0,
    )

    #! ——————— WARNING: the callbacks below do not work with multi-objective —————— !#
    # Custom callback to prune trial if NaN loss is encountered
    nan_pruner_callback = NanLossPrunerCallback(trial)

    # Optuna's built-in pruning callback for early trial termination
    pruning_callback = KerasPruningCallback(trial, monitor)
    #! ———————————————————————————————————————————————————————————————————————————— !#

    # Return the complete list of callbacks
    return [early_stopping, reduce_lr, model_checkpoint, nan_pruner_callback, pruning_callback]

### 4.5. Scalers

In [ ]:
def get_scaler(
    trial: optuna.Trial,
) -> Union[StandardScaler, MinMaxScaler, RobustScaler, QuantileTransformer, PowerTransformer]:
    """
    Suggests and returns a scikit-learn scaler based on Optuna hyperparameter selection.

    Args:
        trial (optuna.Trial): Optuna trial object used to suggest hyperparameters.

    Returns:
        Union[StandardScaler, MinMaxScaler, RobustScaler, QuantileTransformer, PowerTransformer]:
            Instantiated scaler object from scikit-learn.
    """
    # Suggest a scaler name from the list of supported options
    scaler_name = trial.suggest_categorical(
        "scaler",
        [
            "StandardScaler",  # For normally-distributed data
            "MinMaxScaler_-1_1",  # Normalize to [-1, 1] range
            "MinMaxScaler_0_1",  # Normalize to [0, 1] range
            "RobustScaler",  # For data with outliers
            "QuantileTransformer",  # For non-normal or skewed data
            "PowerTransformer",  # For heavy-tailed or skewed data
        ],
    )

    # Return the appropriate scaler instance based on selection
    if scaler_name == "StandardScaler":
        return StandardScaler()
    elif scaler_name == "RobustScaler":
        return RobustScaler()
    elif scaler_name == "QuantileTransformer":
        return QuantileTransformer(output_distribution="normal")
    elif scaler_name == "PowerTransformer":
        return PowerTransformer(method="yeo-johnson")
    elif scaler_name == "MinMaxScaler_0_1":
        return MinMaxScaler(feature_range=(0, 1))
    elif scaler_name == "MinMaxScaler_-1_1":
        return MinMaxScaler(feature_range=(-1, 1))

    # Catch invalid or unknown choices
    else:
        raise ValueError(f"Unknown scaler selected: {scaler_name}")

## 5. Layers Builders

### 5.1. CNN

In [ ]:
def build_cnn1d(
    trial: optuna.Trial,
    x: layers.Layer,
    num_layers: int = 5,
    max_filters: int = 256,
    min_filters: int = 32,
    filter_step: int = 32,
    max_kernel_size: int = 10,
    min_pool_size: int = 2,
    max_pool_size: int = 2,
    use_batch_norm: bool = False,
    use_regularization: bool = False,
    residual_method: Optional[str] = None,
    custom_name: str = "cnn1d",
) -> layers.Layer:
    """
    Builds a 1D CNN where Optuna picks filters, kernel sizes, and pooling window per layer.

    Args:
        trial (optuna.Trial): Optuna trial object.
        x (Layer): Input Keras tensor.
        num_layers (int): Number of Conv1D + Pool1D blocks.
        max_filters (int): Upper bound on number of filters.
        min_filters (int): Lower bound on number of filters.
        filter_step (int): Step size when sampling number of filters.
        max_kernel_size (int): Maximum size of the 1D convolution kernel.
        min_pool_size (int): Minimum size of the 1D pooling window.
        max_pool_size (int): Maximum size of the 1D pooling window.
        use_batch_norm (bool): If True, trial may enable BatchNormalization per layer.
        use_regularization (bool): If True, trial picks kernel, bias, and activity regularizers.
        residual_method (Optional[str]): One of {None, "beside", "all"} to control skip connections.
        custom_name (str): Prefix for naming all layers.

    Returns:
        Layer: Output tensor after applying all CNN blocks.
    """

    # Placeholders for residual connection strategies
    beside_residual: Optional[layers.Layer] = None
    all_skip_connections: List[layers.Layer] = []

    for layer_idx in range(num_layers):
        # 1) Sample pooling window size
        pool_size = trial.suggest_int(
            f"{custom_name}_pool_size_{layer_idx}", min_pool_size, max_pool_size
        )

        # 2) Sample number of filters
        num_filters = trial.suggest_int(
            f"{custom_name}_filters_{layer_idx}",
            min_filters,
            max_filters,
            step=filter_step,
        )

        # 3) Sample convolution kernel size
        kernel_size = trial.suggest_int(
            f"{custom_name}_kernel_size_{layer_idx}", 1, max_kernel_size
        )

        # 4) Activation function
        activation_fn = get_activation(trial, f"{custom_name}_activation_{layer_idx}")

        # 5) Regularizers (if enabled)
        kernel_reg = (
            get_regularizer(trial, f"{custom_name}_kernel_regularizer_{layer_idx}")
            if use_regularization
            else None
        )
        bias_reg = (
            get_regularizer(trial, f"{custom_name}_bias_regularizer_{layer_idx}")
            if use_regularization
            else None
        )
        activity_reg = (
            get_regularizer(trial, f"{custom_name}_activity_regularizer_{layer_idx}")
            if use_regularization
            else None
        )

        # 6) 1D convolution
        x = layers.Conv1D(
            filters=num_filters,
            kernel_size=kernel_size,
            activation=activation_fn,
            padding="same",
            name=f"{custom_name}_conv1d_{layer_idx}",
            kernel_regularizer=kernel_reg,
            bias_regularizer=bias_reg,
            activity_regularizer=activity_reg,
        )(x)

        # 7) Optional BatchNormalization
        if use_batch_norm and trial.suggest_categorical(
            f"{custom_name}_use_batch_norm_{layer_idx}", [True, False]
        ):
            x = layers.BatchNormalization(name=f"{custom_name}_batch_norm_{layer_idx}")(x)

        # 8) Residual connections
        if residual_method == "beside":
            if layer_idx == 0:
                beside_residual = x
            else:
                if trial.suggest_categorical(
                    f"{custom_name}_use_residual_{layer_idx}", [True, False]
                ):
                    prev = beside_residual
                    target_ch = x.shape[-1]
                    # Align channel dimensions if needed
                    if prev.shape[-1] != target_ch:
                        prev = layers.Conv1D(
                            filters=target_ch,
                            kernel_size=1,
                            padding="same",
                            name=f"{custom_name}_res_align_{layer_idx}",
                        )(prev)
                    x = layers.Add(name=f"{custom_name}_res_add_{layer_idx}")([x, prev])
                    beside_residual = x
                else:
                    beside_residual = x

        elif residual_method == "all":
            if layer_idx == 0:
                all_skip_connections = [x]
            else:
                to_add: List[layers.Layer] = []
                for prev_idx, prev_layer in enumerate(all_skip_connections):
                    if trial.suggest_categorical(
                        f"{custom_name}_use_residual_{layer_idx}_{prev_idx}", [True, False]
                    ):
                        prev = prev_layer
                        target_ch = x.shape[-1]
                        if prev.shape[-1] != target_ch:
                            prev = layers.Conv1D(
                                filters=target_ch,
                                kernel_size=1,
                                padding="same",
                                name=f"{custom_name}_skip_res_conv1d_{layer_idx}_{prev_idx}",
                            )(prev)
                        to_add.append(prev)
                if to_add:
                    x = layers.Add(
                        name=f"{custom_name}_res_all_add_{layer_idx}"
                    )([x] + to_add)
                all_skip_connections.append(x)

        # 9) 1D MaxPooling
        x = layers.MaxPooling1D(
            pool_size=pool_size, name=f"{custom_name}_maxpool_{layer_idx}"
        )(x)

    return x

## 6. Objective Function

In [ ]:
def objective(
    trial: optuna.Trial,
    X: List[np.ndarray],
    y: List[np.ndarray],
    checkpoint_dir: str,
    model_dir: str,
    fig_dir: str,
    logs_dir: str,
    epochs: int = 50,
    size_penalizer: Optional[str] = None,
    use_regularization: bool = False,
    residual_method: Optional[str] = None,
    show_summary: bool = False,
    plot_model: bool = False,
) -> float:
    """
    Objective function for Optuna to optimize a Neural Network NN on any-input data.

    Args:
        trial (optuna.Trial): Current trial for hyperparameter suggestions.
        X (List[np.ndarray]): List of input arrays.
        y (List[np.ndarray]): List of label arrays.
        checkpoint_dir (str): Path to store checkpoint files.
        model_dir (str): Path to store full models.
        fig_dir (str): Path to store plots.
        logs_dir (str): Path to store logs.
        epochs (int): Number of training epochs.
        size_penalizer (Optional[str]): type of penalizer to use:
            - "params": Penalizes based on the number of parameters.
            - "flops": Penalizes based on the number of FLOPs.
            - None: No penalization is applied.
        use_regularization (bool): If True, adds regularization (e.g., L1/L2) to layers to prevent overfitting.
        residual_method (Optional[str]): tyoe of residual connection to use:
            - "beside": Adds residual connections between consecutive layers.
            - "all": test residual connections between all layers.
            - None: No residual connections are applied.
        show_summary (bool): If True, display the model summary.
        plot_model (bool): If True, display a plot of the model architecture.

    Returns:
        float: Final validation loss (optionally penalized) used for optimization.
    """

    # Each trial gets a different seed to split the data
    np.random.seed(trial.number)
    tf.random.set_seed(trial.number)

    # ————————————————————————————— Prepare the Data ————————————————————————————— #
    s008_lidar_input = X[0]
    s008_coord_input = X[1]
    s009_lidar_input = X[2]
    s009_coord_input = X[3]
    s008_y_train = y[0]
    s009_y = y[1]

    (
        x_s008_lidar_train,
        x_lidar_val,
        x_s008_coord_train,
        x_coord_val,
        y_s008_train,
        y_val,
    ) = train_test_split(
        s008_lidar_input,
        s008_coord_input,
        s008_y_train,
        test_size=0.2,
        random_state=trial.number,
        shuffle=True,
    )

    (
        x_s009_lidar_test,
        x_s009_lidar_val,
        x_s009_coord_test,
        x_s009_coord_val,
        y_s009_test,
        y_s009_val,
    ) = train_test_split(
        s009_lidar_input,
        s009_coord_input,
        s009_y,
        test_size=0.2,
        random_state=trial.number,
        shuffle=True,
    )

    x_lidar_train = x_s008_lidar_train
    x_coord_train = x_s008_coord_train
    y_train = y_s008_train

    x_lidar_val = np.concatenate((x_lidar_val, x_s009_lidar_val), axis=0)
    x_coord_val = np.concatenate((x_coord_val, x_s009_coord_val), axis=0)
    y_val = np.concatenate((y_val, y_s009_val), axis=0)

    # ———————————————————————————————————————————————————————————————————————————— #

    model = None
    try:

        # ———————————————————————————————————————————————————————————————————————————— #
        #                              Model Construction                              #
        # ———————————————————————————————————————————————————————————————————————————— #

        # ———————————————————— Decide on how to normalize the data ——————————————————— #
        #? These were the best performing options in the previous trials
        lidar_norm_type: str = "none" # trial.suggest_categorical("lidar_norm", ["none", "minmax", "standard"])
        coord_norm_type: str = "standard" # trial.suggest_categorical("coord_norm", ["minmax", "standard"])

        if lidar_norm_type == "minmax":
            lidar_scaler = MinMaxScaler(feature_range=(-1, 1))
        elif lidar_norm_type == "standard":
            lidar_scaler = StandardScaler()
        else:
            lidar_scaler = None

        if coord_norm_type == "minmax":
            coord_scaler = MinMaxScaler(feature_range=(-1, 1))
        elif coord_norm_type == "standard":
            coord_scaler = StandardScaler()

        coord_scaler.fit(x_coord_train)  # Not cheating and using val data
        x_coord_train = coord_scaler.transform(x_coord_train)
        x_coord_val = coord_scaler.transform(x_coord_val)
        x_s009_coord_test = coord_scaler.transform(x_s009_coord_test)
        s009_coord_input = coord_scaler.transform(s009_coord_input)

        if lidar_scaler is not None:
            flat_train = x_s008_lidar_train.reshape(-1, x_s008_lidar_train.shape[-1])
            lidar_scaler.fit(flat_train)  # Not cheating and using val data

            def _scale_lidar(arr: np.ndarray) -> np.ndarray:
                flat = arr.reshape(-1, arr.shape[-1])
                scaled = lidar_scaler.transform(flat)
                return scaled.reshape(arr.shape)

            x_lidar_train = _scale_lidar(x_s008_lidar_train)
            x_lidar_val = _scale_lidar(x_lidar_val)
            x_s009_lidar_test = _scale_lidar(x_s009_lidar_test)
            s009_lidar_input = _scale_lidar(s009_lidar_input)

        # ——————————————————————————————— LiDAR Input ——————————————————————————————— #
        # Input for LiDAR data (e.g., shape: (20, 200, 10))
        x_lidar_input = layers.Input(shape=(20, 200, 10))

        # Inline one-hot encoding of semantic values
        one_hot_lidar = layers.Lambda(
            lambda x: tf.concat(
                [
                    # “Is there a BS anywhere in the 10 channels?” → 1 channel
                    tf.cast(tf.reduce_any(tf.equal(x, -2), axis=-1, keepdims=True), tf.float32),
                    # “Vehicle?” → 1 channel
                    tf.cast(tf.reduce_any(tf.equal(x, -1), axis=-1, keepdims=True), tf.float32),
                    # “Obstacle?” → 1 channel
                    tf.cast(tf.reduce_any(tf.equal(x, 1), axis=-1, keepdims=True), tf.float32),
                    # “Free?” → 1 channel (all channels zero)
                    tf.cast(tf.reduce_all(tf.equal(x, 0), axis=-1, keepdims=True), tf.float32),
                ],
                axis=-1,
            ),
            #! Lambda has deserialization issues, so providing the output shape is necessary
            output_shape=(20, 200, 4),
            name="lidar_one_hot",
        )(x_lidar_input)
        # -> (batch, 20, 200, 4)

        # Merge channels per the trial’s choice
        #? This was the best performing option in the previous trials
        lidar_channel_option: int = 4 # trial.suggest_categorical("lidar_channel_option", [4, 14])
        if lidar_channel_option == 14:
            x_lidar_preproc = layers.Concatenate(name="lidar_concat")(
                [x_lidar_input, one_hot_lidar]
            )  # → (batch,20,200,14)
        else:
            x_lidar_preproc = one_hot_lidar  # → (batch,20,200,4)

        # Flatten the 20×200 grid into a 4000-length sequence with the chosen channels
        x_lidar_flat: layers.Layer = layers.Reshape((20 * 200, lidar_channel_option), name="lidar_flatten")(
            x_lidar_preproc
        )

        # ———————————————————————————————— GPS Input ———————————————————————————————— #
        # Input for coordinate data (e.g., shape: (2,))
        x_coord_input = layers.Input(shape=(2,))

        # Turn (batch,2) → (batch,1,2) → tile to (batch,4000,2)'
        x_coord: layers.Layer = layers.Lambda(
            lambda x: tf.tile(tf.expand_dims(x, axis=1), [1, 20 * 200, 1]),
            #! Lambda has deserialization issues, so providing the output shape is necessary
            output_shape=(20 * 200, 2),
            name="coord_tile_flat",
        )(x_coord_input)

        # ————————————————————————————— Combine Branches ————————————————————————————— #
        # Fuse channels: (batch,20,200,10) + (batch,20,200,2) → (batch,20,200,12)
        combined = layers.Concatenate(axis=-1)([x_lidar_flat, x_coord])

        max_layers = trial.suggest_int("num_layers", 3, 6)

        # Calculate max pool size
        max_pool_dim1 = math.floor((20 * 200) ** (1.0 / max_layers))

        x = build_cnn1d(
            trial,
            combined,
            num_layers=max_layers,
            max_filters=512,
            min_filters=32,
            filter_step=32,
            max_kernel_size=5,
            min_pool_size=1,
            max_pool_size=max_pool_dim1,
            use_batch_norm=True,
            use_regularization=use_regularization,
            residual_method=residual_method,
        )

        # ———————————————————————————— Flatten the Output ———————————————————————————— #
        x = layers.Flatten(name="flatten")(x)

        # ——————————————————————————————— Dense Layers ——————————————————————————————— #
        #? This was the best performing option in the previous trials
        num_dense_layers = 2 # trial.suggest_int("num_dense_layers", 0, 3)
        for i in range(num_dense_layers):
            # Suggest the number of units for each dense layer
            units = trial.suggest_int(f"dense_{i+1}_units", 64, 512, step=64)
            x = layers.Dense(
                units=units,
                activation=get_activation(trial, f"dense_{i+1}_activation"),
                name=f"dense_{i+1}",
            )(x)
            x = layers.Dropout(rate=0.5)(x)

        # —————————————————————————————————— Output —————————————————————————————————— #
        outputs = layers.Dense(256, activation="sigmoid")(x)

        # —————————————————————————— Set Inputs and Outputs —————————————————————————— #
        model = Model(inputs=(x_lidar_input, x_coord_input), outputs=(outputs,))

        # ———————————————————————————— Vizualize the Model ——————————————————————————— #
        if show_summary:
            model.summary()

        if plot_model:
            # Display the model architecture image
            tf.keras.utils.plot_model(
                model,
                to_file=os.path.join(fig_dir, f"model_plot_{trial.number}.png"),
                show_shapes=True,
                show_layer_names=True,
            )
            display(Image(filename=os.path.join(fig_dir, f"model_plot_{trial.number}.png")))

        # ————————————————————————————— Compile the Model ———————————————————————————— #
        optimizer = get_optimizer(trial)
        model.compile(
            optimizer=optimizer,
            loss=losses.BinaryCrossentropy(),
            metrics=["accuracy"],
        )

        # ———————————————————————————————— Train Model ——————————————————————————————— #
        batch_size = 64
        history = model.fit(
            [x_lidar_train, x_coord_train],
            y_train,
            validation_data=([x_lidar_val, x_coord_val], y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=get_callbacks(trial, checkpoint_dir),
            verbose=2,
        )

        model.save(os.path.join(model_dir, f"trial_{trial.number}.keras"))

        # ———————————————————————————————————————————————————————————————————————————— #
        #                            Penalize the Model Size                           #
        # ———————————————————————————————————————————————————————————————————————————— #
        loss = min(history.history["val_loss"])
        if size_penalizer == "flops":
            loss = troo.compute_flops_penalized_loss(loss=loss, model=model)
        elif size_penalizer == "params":
            loss = troo.compute_params_penalized_loss(loss=loss, model=model)

        # ———————————————————————————————————————————————————————————————————————————— #
        #                                 Trial Results                                #
        # ———————————————————————————————————————————————————————————————————————————— #
        clear_output(wait=True)

        epochs = list(range(1, len(history.history["loss"]) + 1))
        train_loss = history.history["loss"]
        val_loss = history.history["val_loss"]
        train_acc = history.history.get("accuracy", [])
        val_acc = history.history.get("val_accuracy", [])
        val_loss_best = min(history.history["val_loss"])

        # Create figure with two subplots
        fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(16, 6))

        # Left: Loss
        ax_loss.plot(epochs, train_loss, marker="o", linestyle="-", label="Training Loss")
        ax_loss.plot(epochs, val_loss, marker="x", linestyle="--", label="Validation Loss")
        ax_loss.set_title("Training & Validation Loss")
        ax_loss.set_xlabel("Epoch")
        ax_loss.set_ylabel("Loss")
        ax_loss.set_xticks(epochs)
        ax_loss.set_ylim(0, max(max(train_loss), max(val_loss)) * 1.05)
        ax_loss.grid(True)
        ax_loss.legend()

        # Right: Accuracy (if available)
        if train_acc and val_acc:
            ax_acc.plot(epochs, train_acc, marker="v", linestyle="-", label="Training Accuracy")
            ax_acc.plot(epochs, val_acc, marker="^", linestyle="--", label="Validation Accuracy")
            ax_acc.set_title("Training & Validation Accuracy")
            ax_acc.set_xlabel("Epoch")
            ax_acc.set_ylabel("Accuracy")
            ax_acc.set_xticks(epochs)
            ax_acc.set_ylim(0, 1)
            ax_acc.grid(True)
            ax_acc.legend()

            trial.set_user_attr("best_train_accuracy", float(max(train_acc)))
            trial.set_user_attr("best_val_accuracy", float(max(val_acc)))
        else:
            ax_acc.axis("off")  # hide if accuracy not present

        fig.tight_layout()
        fig.savefig(os.path.join(fig_dir, f"trial_{trial.number}.png"), dpi=300)
        plt.close(fig)

        # ————————————————————————————— Evaluate on s009 ————————————————————————————— #
        test_loss, test_acc = model.evaluate(
            [x_s009_lidar_test, x_s009_coord_test], y_s009_test, batch_size=batch_size, verbose=0
        )

        trial.set_user_attr("test_accuracy_s009", float(test_acc))

        # Now evaluate on the full s009 dataset for comparison purposes
        test_loss_full, test_acc_full = model.evaluate(
            [s009_lidar_input, s009_coord_input], s009_y, batch_size=batch_size, verbose=0
        )
        trial.set_user_attr("test_accuracy_s009_full", float(test_acc_full))

        # ————————————————————————————— Print the results ———————————————————————————— #

        print(f"\n\n# ——————————————————————— Trial {trial.number} Results ——————————————————————— #")
        print("\n" + "=" * 15)
        print(f"Training loss: {loss:.12f}")
        print(f"Training accuracy: {max(train_acc):.4f}\n")
        print(f"Validation loss: {val_loss_best:.12f}")
        print(f"Validation accuracy: {max(val_acc):.4f}\n")
        print(f"Test loss (s009): {test_loss:.12f}")
        print(f"Test accuracy (s009): {test_acc:.4f}\n")
        print(f"Test loss (s009 full): {test_loss_full:.12f}")
        print(f"Test accuracy (s009 full): {test_acc_full:.4f}\n")

        params = model.count_params()
        print(f"Number of parameters: {params}")
        print(f"Model size: {params * 4 / (1024 ** 2):.2f} MB")
        print("=" * 15 + "\n")
        print("# ———————————————————————————————————————————————————————————————————————————— #\n\n")
        
        trial.set_user_attr("num_params", params)
        trial.set_user_attr("model_size_mb", params * 4 / (1024 ** 2))

        return loss

    except optuna.exceptions.TrialPruned:
        raise  # simply propagate pruning
    except tf.errors.ResourceExhaustedError as oom_err:
        # Catch OOM / resource exhausted
        print(f"❌ Trial {trial.number} hit OOM (ResourceExhaustedError): {oom_err}")

        # Log the error to a file in the logs directory
        error_log_path = os.path.join(logs_dir, f"trial_{trial.number}_error.log")
        with open(error_log_path, "w") as log_file:
            log_file.write(f"Trial {trial.number} encountered an error:\n")
            log_file.write(str(oom_err) + "\n\n")
            log_file.write("Traceback:\n")
            traceback.print_exc(file=log_file)

        return float("inf")  # Return bad loss
    except Exception as e:
        print(f"An error occurred during the trial execution: {e}")
        traceback.print_exc()

        # Log the error to a file in the logs directory
        error_log_path = os.path.join(logs_dir, f"trial_{trial.number}_error.log")
        with open(error_log_path, "w") as log_file:
            log_file.write(f"Trial {trial.number} encountered an error:\n")
            log_file.write(str(e) + "\n\n")
            log_file.write("Traceback:\n")
            traceback.print_exc(file=log_file)

        return float("inf")  # Return bad loss
    finally:
        if model is not None:
            clear_session()
            del model

## 7. Code Health Check

In [ ]:
# resources_dir = os.path.join(RUN_DIR, "resources")
# os.makedirs(resources_dir, exist_ok=True)
# troo.log_resources(log_dir=resources_dir)

In [ ]:
try:
    pid = os.getpid()
    cmd = (
        f'python3 "{os.path.abspath("_monitor_kernel_life.py")}" '
        f"--pid {pid} --custom-title {RUN_DIR}; exec bash"
    )
    terminals = [
        ["xfce4-terminal", "--disable-server", "--hold", "-e", f'bash -c "{cmd}"'],
        ["gnome-terminal", "--disable-factory", "--", "bash", "-i", "-c", cmd],
        ["xterm", "-hold", "-e", cmd],
        ["konsole", "--hold", "-e", f'bash -c "{cmd}"'],
    ]
    term = next((t for t in terminals if shutil.which(t[0])), None)
    if not term:
        raise RuntimeError(
            "No supported terminal emulator found; install gnome-terminal, "
            "xfce4-terminal, konsole, or xterm."
        )
    _monitor_proc = subprocess.Popen(term, preexec_fn=os.setpgrp)
    print(f"[INFO] Launched monitor in {term[0]} (PID={pid})")
except Exception as e:
    print(f"[ERROR] Auto launching kernel monitoring failed! {e}\n")
    display(
        HTML(
            f"Call the monitor script manually: "
            f'<span style="color: orange;">'
            f"python _monitor_kernel_life.py --pid {pid} --custom-title {RUN_DIR}"
            f"</span>"
        )
    )
    pass

## Main

In [ ]:
try:
    # ——————————————————————————————— Storage paths —————————————————————————————— #
    study_dir = os.path.join(RUN_DIR, "optuna_study")
    os.makedirs(study_dir, exist_ok=True)

    dirs = {
        "args": os.path.join(study_dir, "args"),
        "figures": os.path.join(study_dir, "figures"),
        "weights": os.path.join(study_dir, "weights"),
        "models": os.path.join(study_dir, "models"),
        "logs": os.path.join(study_dir, "logs"),
    }
    for path in dirs.values():
        os.makedirs(path, exist_ok=True)

    storage_path = f"sqlite:///{os.path.join(study_dir, 'optuna_study.db')}"
    checkpoint_dir, model_dir, fig_dir, args_dir, logs_dir = (
        dirs["weights"],
        dirs["models"],
        dirs["figures"],
        dirs["args"],
        dirs["logs"],
    )

    print(f"Initializing study at '{study_dir}'...")

    # —————————————————————————————————— Pruners ————————————————————————————————— #
    pruner = optuna.pruners.HyperbandPruner()

    # ——————————————————————————————————— Study —————————————————————————————————— #
    study = optuna.create_study(
        study_name=os.path.basename(study_dir),
        storage=storage_path,
        direction="minimize",
        pruner=pruner,
        load_if_exists=True,
    )

    # Count trials done, then determine the remaining trials
    done_trials = len(
        study.get_trials(
            deepcopy=False,
            states=(
                optuna.trial.TrialState.COMPLETE,
                optuna.trial.TrialState.PRUNED,
                optuna.trial.TrialState.FAIL,
            ),
        )
    )
    n_remaining_trials = max(0, NUM_TRIALS - done_trials)

    study.optimize(
        lambda trial: objective(
            trial,
            X=[s008_lidar_input, s008_coord_input, s009_lidar_input, s009_coord_input],
            y=[s008_y_train, s009_y],
            checkpoint_dir=checkpoint_dir,
            model_dir=model_dir,
            fig_dir=fig_dir,
            logs_dir=logs_dir,
            epochs=EPOCHS,
            size_penalizer=None,
            use_regularization=False,
            residual_method=None,  #! Find your backbone first
            show_summary=False,
        ),
        n_trials=n_remaining_trials,
        catch=(ValueError, RuntimeError),
        gc_after_trial=True,
        n_jobs=1,  # If you have multiple GPUs/Cores
        show_progress_bar=False,
    )

    # ————————————————————————————— Save Top-K Trials ———————————————————————————— #
    valid_trials = [
        t for t in study.trials if t.value is not None and not (math.isnan(t.value) or math.isinf(t.value))
    ]
    sorted_trials = sorted(valid_trials, key=lambda t: t.value)[:TOP_K]

    for rank, trial in enumerate(sorted_trials):
        trial_id = trial.number
        trial_params = trial.params
        trial_loss = trial.value
        trial_train_acc = trial.user_attrs.get("best_train_accuracy", None)
        trial_val_acc = trial.user_attrs.get("best_val_accuracy", None)
        trial_test_acc = trial.user_attrs.get("test_accuracy_s009", None)
        trial_test_acc_full = trial.user_attrs.get("test_accuracy_s009_full", None)
        trial_num_params = trial.user_attrs.get("num_params", None)
        trial_model_size = trial.user_attrs.get("model_size_mb", None)

        troo.save_trial_params_to_file(
            filepath=os.path.join(args_dir, f"top_{rank + 1}_trial.txt"),
            params=trial_params,
            rank=rank + 1,
            trial_id=trial_id,
            loss=trial_loss,
            val_accuracy=trial_val_acc,
            train_accuracy=trial_train_acc,
            test_accuracy=trial_test_acc,
            test_accuracy_full=trial_test_acc_full,
            num_params=trial_num_params,
            model_size_mb=trial_model_size,
            sampler=study.sampler.__class__.__name__,
        )

    # —————————————————————————— Clean-Up Non-Top Trials ————————————————————————— #
    all_trial_ids = {t.number for t in study.trials}
    top_trial_ids = {t.number for t in sorted_trials}

    cleanup_paths = [
        (checkpoint_dir, "trial_{trial_id}.weights.h5"),
        (model_dir, "trial_{trial_id}.keras"),
        (fig_dir, "trial_{trial_id}.png"),
    ]

    for trial_id in all_trial_ids - top_trial_ids:
        for base_dir, filename_template in cleanup_paths:
            file_path = os.path.join(base_dir, filename_template.format(trial_id=trial_id))
            if os.path.exists(file_path):
                os.remove(file_path)

    troo.analyze_study(study, fig_dir=fig_dir, table_dir=study_dir)

    # ————————————————————————————— End The Training ————————————————————————————— #
    failed_trials = sum(
        1
        for t in study.trials
        if t.state not in {optuna.trial.TrialState.COMPLETE, optuna.trial.TrialState.PRUNED}
    )
    notify_training_success(
        recipients_file="./json/recipients.json",
        credentials_file="./json/credentials.json",
        subject=f"🎉 {RUN_DIR} Training Complete - Failed Trials: {failed_trials}",
    )
except Exception as e:
    print(f"An error occurred: {e}")
    traceback.print_exc()

In [ ]:
# Kill the monitor kernel life process
if _monitor_proc is not None and _monitor_proc.poll() is None:
    os.killpg(_monitor_proc.pid, signal.SIGINT)